# Aşama 5: Hassasiyet, keşif potansiyeli ve akı üst sınırları

Şimdiye kadar "**bulduğumuz şey** ne kadar anlamlı?" diye sorduk. Bu aşamada tersini soruyoruz: "**analizimiz neyi bulabilir?**"

Üç kavram:

| Kavram | Soru | Tanım (IceCube standardı) |
|---|---|---|
| **Hassasiyet** (sensitivity) | Kaynak gerçekten varsa, ne kadar parlak olmalı ki genelde fark edelim? | Denemelerin **%90'ında** TS, sadece-arka-plan TS'lerinin medyanını geçiyor |
| **Keşif potansiyeli** | Ne kadar parlak olmalı ki 5σ keşif yapalım? | Denemelerin **%50'sinde** TS, 5σ eşiğini geçiyor |
| **Üst sınır** (upper limit) | Bulduğumuz TS'ye bakarak kaynak **en fazla** ne kadar parlak olabilir? | Denemelerin **%90'ında** TS, gözlenen TS'yi geçiyor |

Hepsini aynı yöntemle hesaplıyoruz: arka plana bilinen miktarda **sahte sinyal** ekleyip analizi binlerce kez tekrar çalıştırıyoruz.

Son adımda nötrino **sayısını** fiziksel bir **akıya** (cm² başına saniyede kaç nötrino) çeviriyoruz. Bunun için dedektörün tepki dosyalarını kullanacağız. Böylece sonuçlarımızı IceCube makaleleriyle ve teorik modellerle aynı birimde karşılaştırabileceğiz.

In [ ]:
import sys; sys.path.append('../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import importlib
import icecube_ps as ps
importlib.reload(ps)        # src/ içindeki dosya değiştiyse güncel hâlini yükle
assert float(ps.SURUM) >= float('5.1'), (
    f"src/icecube_ps.py eski sürüm ({getattr(ps, 'SURUM', 'yok')}). "
    "Yeni dosyayı src/ klasörüne kopyalayıp kernel'i yeniden başlat.")
print('icecube_ps sürümü:', ps.SURUM, '|', ps.__file__)

pd.set_option('display.float_format', '{:.3g}'.format)   # 1.82e-10 gibi okunur sayılar
DATA, RESULTS = Path('../data'), Path('../results')
ev = ps.veri_yukle(DATA)
B = ps.ArkaPlan(ev['dec'].values)
asama3 = pd.read_csv(RESULTS / 'asama3_likelihood.csv').set_index('kaynak')

def dosya_adi(ad):
    return ad.replace(' ', '_').replace('+', 'p')

ts_arka = {ad: np.load(RESULTS / f'ts_arka_{dosya_adi(ad)}.npy') for ad in ps.KAYNAKLAR}
asama3[['ns', 'TS', 'sigma_Wilks']]

## 1. Dedektör tepkisi: Etkin alan

IceCube'un "ne kadar büyük" olduğu enerjiye ve yöne bağlı. **Etkin alan** $A_{\text{eff}}(E, \delta)$ şu soruyu cevaplıyor: bu enerjide, bu yönden gelen nötrinoları yakalama açısından dedektör kaç cm²'lik mükemmel bir dedektöre eşdeğer?

Nötrinolar çok nadir etkileştiği için 1 km³'lük dedektörün etkin alanı düşük enerjilerde sadece birkaç cm². Enerji arttıkça etkileşme olasılığı ve müon menzili arttığı için hızla büyüyor.

**Canlı süre:** Dedektörün analize uygun veri aldığı toplam süre (bakım, kalibrasyon vb. hariç).

In [ ]:
alanlar = ps.etkin_alanlari_yukle(DATA)
sureler = ps.canli_sureler_yukle(DATA)
print('Etkin alan dosyaları:', list(alanlar))
print()
for s, T in sureler.items():
    print(f"{s:10s} canlı süre: {T / 86400:7.1f} gün")
print(f"{'TOPLAM':10s} canlı süre: {sum(sureler.values()) / 86400 / 365.25:.2f} yıl")

# Biçim kontrolü: dosyanın ilk satırları ve başlığı
ornek = next(ps._metin_dosyalari(DATA, 'irfs/*effectiveArea*'))
print('\n', ornek.name); print(open(ornek, encoding='latin-1').read()[:400])

In [ ]:
son_sezon = sorted(alanlar)[-1]          # en güncel dedektör konfigürasyonu
a = alanlar[son_sezon]
fig, ax = plt.subplots(figsize=(9, 5))
for dec, renk in [(-30, '#d1495b'), (0, '#edae49'), (30, '#66a182'), (75, '#2e4057')]:
    s = a[(a['dec_min'] <= dec) & (dec < a['dec_max'])]
    ax.step(s['logE_min'], s['A_cm2'], where='post', color=renk, lw=2, label=f'Dec = {dec}°')
ax.set_yscale('log')
ax.set_xlabel('log₁₀(Eν / GeV)'); ax.set_ylabel('A_eff [cm²]')
ax.set_title(f'IceCube etkin alanı ({son_sezon})')
ax.legend(); ax.grid(alpha=0.3, which='both')
plt.savefig(RESULTS / 'asama5_etkin_alan.png', dpi=150, bbox_inches='tight')
plt.show()

**Grafikte ara:**
- **Güney (−30°):** Düşük enerjide etkin alan çok küçük. Aşama 1'deki sert enerji kesiminin dedektör tepkisindeki izi.
- **Kuzey kutbu (+75°):** En yüksek enerjilerde etkin alan **düşüyor**. Aşama 1'de enerji-yön grafiğinde gördüğümüz Dünya soğurması: çok yüksek enerjili nötrinolar Dünya'nın çekirdeğinden geçemiyor.

Aynı fiziği şimdi üçüncü kez, bu sefer simülasyon tablolarından görüyoruz.

## 2. Nötrino sayısından akıya

Kaynaktan gelen nötrino akısının bir **kuvvet yasası** olduğunu varsayıyoruz (kozmik hızlandırıcılar için standart varsayım):

$$\Phi(E) = \Phi_0 \left(\frac{E}{1\,\text{TeV}}\right)^{-\gamma}$$

Beklenen olay sayısı, akıyla etkin alanın çarpımının enerji üzerinden integrali ve canlı süreyle çarpımı:

$$N = \sum_{\text{sezon}} T_{\text{sezon}} \int A_{\text{eff}}(E,\delta)\,\Phi(E)\,dE$$

Bu bağıntı $\Phi_0$'da doğrusal. $\Phi_0 = 1$ için $N$'yi bir kere hesaplarsak, herhangi bir nₛ'i akıya çevirmek bir bölme işlemi: $\Phi_0 = n_s / N_1$.

Sonuçları literatürdeki standart birimle veriyoruz: $E_0^2\,\Phi(E_0)$ [TeV cm⁻² s⁻¹], $E_0$ = 1 TeV. **γ = 2** standart referans; **γ = 3** NGC 1068 için daha gerçekçi (IceCube bu kaynağın spektrumunu oldukça yumuşak buldu).

In [ ]:
donusum = []
for ad, (ra_k, dec_k) in ps.KAYNAKLAR.items():
    donusum.append(dict(kaynak=ad,
                        N1_gamma2=ps.aki_basina_olay(dec_k, 2.0, alanlar, sureler),
                        N1_gamma3=ps.aki_basina_olay(dec_k, 3.0, alanlar, sureler)))
donusum = pd.DataFrame(donusum).set_index('kaynak')
donusum

## 3. Sinyal denemeleri

Her kaynak için, ortalama nₛ'i 0'dan yukarı doğru artırarak şunu yapıyoruz:
1. Arka planı karıştır (Aşama 4'teki gibi).
2. Poisson sayıda sahte sinyal olayı ekle. (Ortalama 20 diyorsak bir denemede 17, öbüründe 24 olabilir; gerçek gökyüzünde de böyle.)
3. Analizi çalıştır, TS'yi kaydet.

Sahte olayların açısal belirsizliklerini kaynağın Dec'ine yakın gerçek olaylardan çekiyoruz (Aşama 3'teki ders).

⏱️ Kaynak başına birkaç dakika. Sonuçlar kaydediliyor.

In [ ]:
NS_IZGARA = np.arange(0, 129, 8)
N_DENEME = 300
rng = np.random.default_rng(2026)

modeller, sinyal_ts = {}, {}
for ad, (ra_k, dec_k) in ps.KAYNAKLAR.items():
    L = ps.NoktaKaynakLikelihood(ev, B, ra_k, dec_k)
    L.sinyal_havuzu_ayarla(ev)
    modeller[ad] = L
    dosya = RESULTS / f'ts_sinyal_{dosya_adi(ad)}.npy'
    if dosya.exists():
        sinyal_ts[ad] = np.load(dosya)
    else:
        print(f'{ad}: sinyal denemeleri...', flush=True)
        sinyal_ts[ad] = np.array([L.sinyal_denemeleri(n, N_DENEME, rng)[:, 1]
                                  for n in NS_IZGARA])
        np.save(dosya, sinyal_ts[ad])
print('Tamam')

## 4. Üç eşik, üç eğri

NGC 1068 örneğinde: yatay eksen eklenen ortalama sinyal, dikey eksen "TS'si eşiği geçen denemelerin oranı". Sinyal arttıkça her eğri 0'dan 1'e tırmanıyor. Her kavram, ilgili eğrinin ilgili orana (%90 veya %50) ulaştığı nokta.

In [ ]:
ad = 'NGC 1068'
ts_a = ts_arka[ad]
eta = np.mean(ts_a > 0)
esikler = {
    'Hassasiyet: TS > arka plan medyanı':  (np.median(ts_a), 0.9, '#66a182'),
    'Keşif: TS > 5σ eşiği':                 (ps.ts_5sigma(eta), 0.5, '#d1495b'),
    f"Üst sınır: TS > gözlenen ({asama3.loc[ad, 'TS']})": (asama3.loc[ad, 'TS'], 0.9, '#2e4057'),
}
fig, ax = plt.subplots(figsize=(9, 5))
for etiket, (esik, oran, renk) in esikler.items():
    ns_x, f = ps.esik_gecen_ns(NS_IZGARA, sinyal_ts[ad], esik, oran)
    ax.plot(NS_IZGARA, f, 'o-', color=renk, label=etiket)
    ax.axhline(oran, color=renk, ls=':', lw=1)
    if np.isfinite(ns_x):
        ax.axvline(ns_x, color=renk, ls='--', lw=1)
ax.set_xlabel('eklenen ortalama sinyal nₛ'); ax.set_ylabel('eşiği geçen deneme oranı')
ax.set_title(f'{ad}: hassasiyet tanımları')
ax.legend(fontsize=9, loc='lower right'); ax.grid(alpha=0.3)
plt.savefig(RESULTS / 'asama5_esik_egrileri.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
satirlar = []
for ad in ps.KAYNAKLAR:
    ts_a, ts_s = ts_arka[ad], sinyal_ts[ad]
    eta = np.mean(ts_a > 0)
    ns_hass, _ = ps.esik_gecen_ns(NS_IZGARA, ts_s, np.median(ts_a), 0.9)
    ns_kesif, _ = ps.esik_gecen_ns(NS_IZGARA, ts_s, ps.ts_5sigma(eta), 0.5)
    ns_ust, _ = ps.esik_gecen_ns(NS_IZGARA, ts_s, asama3.loc[ad, 'TS'], 0.9)
    N1 = donusum.loc[ad, 'N1_gamma2']
    satirlar.append(dict(
        kaynak=ad, ns_bulunan=asama3.loc[ad, 'ns'],
        ns_hassasiyet=round(ns_hass, 1), ns_kesif=round(ns_kesif, 1), ns_ust_sinir=round(ns_ust, 1),
        E2F_bulunan=ps.e2_aki(asama3.loc[ad, 'ns'] / N1),
        E2F_hassasiyet=ps.e2_aki(ns_hass / N1),
        E2F_kesif=ps.e2_aki(ns_kesif / N1),
        E2F_ust_sinir=ps.e2_aki(ns_ust / N1)))
tablo5 = pd.DataFrame(satirlar).set_index('kaynak')
tablo5.to_csv(RESULTS / 'asama5_hassasiyet.csv')
tablo5

`E2F_*` sütunları γ = 2 için $E_0^2\Phi(E_0)$, TeV cm⁻² s⁻¹ cinsinden.

**Nasıl okunur:** Bir kaynağın `ns_bulunan` değeri `ns_kesif`'e yaklaşıyorsa, analizimiz o kaynağı keşfetmenin eşiğinde demektir. `ns_bulunan`, `ns_hassasiyet`'in altındaysa bulduğumuz sinyal gürültüden ayırt edilemeyecek kadar zayıf.

## 5. Gökyüzü boyunca hassasiyet

IceCube makalelerinin klasik grafiği: hassasiyet ve keşif potansiyeli, yükselime (Dec) göre nasıl değişiyor? Bunun için gökyüzünün farklı yükselimlerinde boş test noktaları seçip her birinde arka plan + sinyal denemelerini tekrarlıyoruz.

⏱️ En uzun hücre bu (10–15 dakika). Sonuçlar kaydediliyor.

In [ ]:
DEC_IZGARA = [-60, -30, -15, -5, 0, 5, 15, 30, 45, 60, 75]
dosya = RESULTS / 'asama5_dec_hassasiyet.csv'
if dosya.exists():
    egri = pd.read_csv(dosya)
else:
    rng = np.random.default_rng(11)
    izgara = np.arange(0, 129, 8)
    satirlar = []
    for dec in DEC_IZGARA:
        print(f'Dec = {dec}°...', flush=True)
        L = ps.NoktaKaynakLikelihood(ev, B, 123.4, dec)    # RA önemsiz: karıştırıyoruz
        L.sinyal_havuzu_ayarla(ev)
        ts_a = L.arka_plan_denemeleri(2000, rng)
        ts_s = np.array([L.sinyal_denemeleri(n, 150, rng)[:, 1] for n in izgara])
        ns_h, _ = ps.esik_gecen_ns(izgara, ts_s, np.median(ts_a), 0.9)
        ns_k, _ = ps.esik_gecen_ns(izgara, ts_s, ps.ts_5sigma(np.mean(ts_a > 0)), 0.5)
        N1 = ps.aki_basina_olay(dec, 2.0, alanlar, sureler)
        satirlar.append(dict(dec=dec, ns_hass=ns_h, ns_kesif=ns_k,
                             E2F_hass=ps.e2_aki(ns_h / N1), E2F_kesif=ps.e2_aki(ns_k / N1)))
    egri = pd.DataFrame(satirlar)
    egri.to_csv(dosya, index=False)
egri

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x = np.sin(np.radians(egri['dec']))
ax.plot(x, egri['E2F_hass'], '-', color='#66a182', lw=2.5, label='Hassasiyet (%90)')
ax.plot(x, egri['E2F_kesif'], '--', color='#d1495b', lw=2.5, label='5σ keşif potansiyeli (%50)')

renkler = ['#2e4057', '#edae49', '#7a8fb8', '#8f5fa8']
for (ad, satir), renk in zip(tablo5.iterrows(), renkler):
    xs = np.sin(np.radians(ps.KAYNAKLAR[ad][1]))
    ax.plot(xs, satir['E2F_ust_sinir'], 'v', color=renk, ms=11, label=f'{ad}: %90 üst sınır')
    ax.plot(xs, satir['E2F_bulunan'], 'o', color=renk, ms=8, mfc='none', mew=2)
ax.plot([], [], 'o', color='gray', mfc='none', mew=2, label='en iyi fit akısı (○)')

ax.set_yscale('log')
ax.set_xlabel('sin(Dec)'); ax.set_ylabel('E² Φ  @ 1 TeV  [TeV cm⁻² s⁻¹]   (γ = 2)')
ax.set_title('IceCube 10 yıl: nokta kaynak hassasiyeti (bu analiz)')
ax.axvline(np.sin(np.radians(-5)), color='gray', ls=':', lw=1)
ax.text(np.sin(np.radians(-5)) - 0.02, ax.get_ylim()[1], ' güney | kuzey', ha='center', va='top',
        fontsize=9, color='gray')
ax.legend(fontsize=8.5, loc='upper right'); ax.grid(alpha=0.3, which='both')
plt.savefig(RESULTS / 'asama5_hassasiyet_dec.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. NGC 1068'in akısı

En iyi fit nₛ'i ve 1σ aralığını akıya çeviriyoruz: "**NGC 1068'den gelen nötrino akısını ölçtük**" diyebileceğimiz sayı. γ = 3 için de veriyoruz; bu kaynak için daha gerçekçi spektrum o.

In [ ]:
ad = 'NGC 1068'
alt, ust = [float(x) for x in asama3.loc[ad, 'ns_1sigma'].replace('–', '-').split('-')]
for gamma in (2.0, 3.0):
    N1 = ps.aki_basina_olay(ps.KAYNAKLAR[ad][1], gamma, alanlar, sureler)
    f = lambda n: ps.e2_aki(n / N1)
    print(f"γ = {gamma:.0f}:  E²Φ(1 TeV) = {f(asama3.loc[ad, 'ns']):.2e}  "
          f"[{f(alt):.2e} – {f(ust):.2e}]  TeV cm⁻² s⁻¹")

## Yorumlarken

- **Hassasiyet eğrisinin şekli:** Kuzeyde (sin Dec > 0) eğri düşük: Dünya filtresi sayesinde arka plan temiz, zayıf kaynakları bile görebiliyoruz. Güneyde eğri **onlarca kat** yüksek: arka plan sert kesimle temizlendiği için düşük enerjili sinyali de kaybediyoruz. Aşama 1'deki fiziğin nicel sonucu.
- **Üst sınırlar neden hassasiyetin üstünde?** Bu kaynaklarda fazlalık bulduk. Fazlalık bulunca "kaynak en fazla şu kadar parlak olabilir" sınırı yukarı kayar. Fazlalık bulunmayan bir kaynakta üst sınır hassasiyetin civarında olurdu.
- **Neden "%90" ve "%50"?** Bunlar alanın uzlaşılmış tanımları. Kendi sonuçlarımızı IceCube makaleleriyle aynı dilde konuşturmak için aynı tanımları kullanıyoruz.
- **Sadeleştirmelerimiz:** Enerji bilgisini likelihood'da kullanmıyoruz ve sahte sinyalin açısal belirsizliklerini gerçek olaylardan çekiyoruz (gerçek sinyal genelde daha yüksek enerjili, dolayısıyla yönü daha kesin). İkisi de bizi **IceCube'dan daha az hassas** yapıyor: eğrimizin makaledekinin birkaç kat üstünde çıkması beklenen bir sonuç. Aşama 9'da enerji terimini ekleyince bu farkın kapanmasını bekliyoruz.